# RAG Chat System — Colab Notebook

This notebook runs the **Retrieval-Augmented Generation (RAG)** pipeline from the `rag-project` in Google Colab.

## Steps
1. Install dependencies
2. Set your API key
3. Upload documents
4. Initialize the RAG pipeline
5. Ask questions interactively

---
## 1. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

---
## 2. Set Your API Key

**Option A** — Use Colab Secrets (recommended):  
Add `OPENAI_API_KEY` in the 🔑 Secrets panel on the left sidebar.

**Option B** — Set it directly in code (less secure):

In [ ]:
import os

# Option A: Colab Secrets (uncomment if using Secrets panel)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Option B: Paste your key directly (uncomment and fill in)
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# If using a custom OpenAI-compatible server (e.g. LM Studio), set the base URL:
# os.environ["OPENAI_API_BASE_URL"] = "http://localhost:1234/v1"

print("API key configured:", "OPENAI_API_KEY" in os.environ)

---
## 3. Upload Documents

Upload `.txt`, `.pdf`, or `.md` files to the `documents/` folder.  
You can either:
- Use the file upload widget below, or
- Manually upload files to the `documents/` directory in the Colab file browser.

In [ ]:
import os
from google.colab import files

DOCUMENTS_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "documents")
os.makedirs(DOCUMENTS_DIR, exist_ok=True)

print(f"Upload your documents (.txt, .pdf, .md) — they will be saved to: {DOCUMENTS_DIR}")
uploaded = files.upload()

for filename, content in uploaded.items():
    filepath = os.path.join(DOCUMENTS_DIR, filename)
    with open(filepath, "wb") as f:
        f.write(content)
    print(f"  Saved: {filepath}")

print(f"\nDocuments in folder: {os.listdir(DOCUMENTS_DIR)}")

---
## 4. Initialize the RAG Pipeline

This loads documents, creates embeddings, builds the FAISS vector store, and assembles the LangChain conversational chain.

In [ ]:
import sys
import os

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
pipeline.initialize()

print("\n✅ Pipeline is ready! You can now ask questions below.")

---
## 5. Ask Questions

Run the cell below to ask a single question, or use the interactive loop cell after it.

In [ ]:
# --- Single Question ---
question = "What are the main topics covered in the documents?"  # <-- Edit your question here

response = pipeline.ask(question)
print(f"Q: {question}")
print(f"A: {response}")

In [ ]:
# --- Interactive Chat Loop ---
# Type your questions one at a time. Type 'quit' to stop.

print("Interactive RAG Chat (type 'quit' to stop, 'clear' to reset history)")
print("=" * 60)

while True:
    question = input("\nYou: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if question.lower() == "clear":
        pipeline.clear_history()
        print("History cleared.")
        continue

    response = pipeline.ask(question)
    print(f"\nAssistant: {response}")

---
## 6. Direct Document Search (Optional)

Search the vector store directly without going through the full RAG chain.

In [ ]:
query = "supplier sustainability"  # <-- Edit your search query here

results = pipeline.search_documents(query)
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:400])